In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ── 1. Download dataset automatically via kagglehub ──────────────────────────
# This uses Colab's cache if already downloaded — no manual upload needed
print("Downloading dataset...")
data_path = kagglehub.dataset_download('mrwellsdavid/unsw-nb15')
print(f"Dataset ready at: {data_path}")

# List available files so we know exact filenames
print("\nFiles in dataset:")
for f in sorted(os.listdir(data_path)):
    print(f"  {f}")

# ── 2. Config ─────────────────────────────────────────────────────────────────
OUTPUT_DIR = '/content/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

COL_NAMES = [
    'srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur',
    'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service',
    'sload', 'dload', 'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb',
    'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit',
    'stime', 'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat',
    'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login',
    'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm',
    'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm',
    'attack_cat', 'Label'
]

CATEGORICAL_COLS = ['proto', 'service', 'state']
TARGET_COL       = 'attack_cat'

# ── 3. Preprocessing functions ────────────────────────────────────────────────
def clean_data(df):
    df = df.copy()
    df['attack_cat'] = (
        df['attack_cat']
        .fillna('normal')
        .str.strip()
        .str.lower()
        .replace('backdoors', 'backdoor')
    )
    df['ct_flw_http_mthd'] = df['ct_flw_http_mthd'].fillna(0)
    df['is_ftp_login']     = df['is_ftp_login'].fillna(0)
    df['is_ftp_login']     = np.where(df['is_ftp_login'] > 1, 1, df['is_ftp_login'])
    df['service']          = df['service'].replace('-', 'None')
    df['ct_ftp_cmd']       = df['ct_ftp_cmd'].replace(' ', 0).astype(int)
    return df

def select_features(df):
    return df.drop(columns=['srcip', 'sport', 'dstip', 'dsport', 'Label'])

def downcast_dtypes(df):
    for col in df.select_dtypes(include='float64').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include='int64').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

# ── 4. Encoder/scaler state (fitted on first file, reused on rest) ────────────
label_encoders = {}
scaler         = None
numeric_cols   = None

def fit_encoders(df):
    global scaler, numeric_cols
    numeric_cols = [c for c in df.columns if c not in CATEGORICAL_COLS + [TARGET_COL]]
    for col in CATEGORICAL_COLS:
        le = LabelEncoder()
        le.fit(df[col].astype(str))
        label_encoders[col] = le
    scaler = StandardScaler()
    scaler.fit(df[numeric_cols])
    print("  Encoders and scaler fitted.")

def encode_and_normalize(df):
    df = df.copy()
    for col in CATEGORICAL_COLS:
        le = label_encoders[col]
        df[col] = df[col].astype(str).apply(
            lambda x: x if x in le.classes_ else le.classes_[0]
        )
        df[col] = le.transform(df[col])
    df[numeric_cols] = scaler.transform(df[numeric_cols])
    return df

# ── 5. Process files one at a time → stream to output ────────────────────────
csv_out  = os.path.join(OUTPUT_DIR, 'unsw_nb15_processed.csv')
json_out = os.path.join(OUTPUT_DIR, 'unsw_nb15_processed.json')

for f in [csv_out, json_out]:
    if os.path.exists(f):
        os.remove(f)

first_file = True

for i in range(1, 5):
    path = os.path.join(data_path, f'UNSW-NB15_{i}.csv')
    print(f"\nProcessing file {i}/4...")

    df = pd.read_csv(path, header=None, names=COL_NAMES)
    print(f"  Loaded: {len(df):,} rows")

    df = clean_data(df)
    df = select_features(df)
    df = downcast_dtypes(df)

    if first_file:
        fit_encoders(df)

    df = encode_and_normalize(df)

    df.to_csv(csv_out,  mode='a', header=first_file, index=False)
    df.to_json(json_out, orient='records', lines=True, mode='a')

    print(f"  ✅ Done — {len(df):,} rows written")
    del df
    first_file = False

print(f"\n{'='*50}")
print(f"✅ Processing complete!")
print(f"   CSV  → {csv_out}")
print(f"   JSON → {json_out}")

# ── 6. Download the output files to your machine ─────────────────────────────
from google.colab import files
print("\nStarting download...")
files.download(csv_out)
files.download(json_out)

Using Colab cache for faster access to the 'unsw-nb15' dataset.
Dataset ready at: /kaggle/input/unsw-nb15

Files in dataset:
  NUSW-NB15_features.csv
  UNSW-NB15_1.csv
  UNSW-NB15_2.csv
  UNSW-NB15_3.csv
  UNSW-NB15_4.csv
  UNSW-NB15_LIST_EVENTS.csv
  UNSW_NB15_testing-set.csv
  UNSW_NB15_training-set.csv

Processing file 1/4...


/tmp/ipykernel_5533/766213145.py:105: DtypeWarning: Columns (1,3,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, header=None, names=COL_NAMES)


  Loaded: 700,001 rows
  Encoders and scaler fitted.
  ✅ Done — 700,001 rows written

Processing file 2/4...


/tmp/ipykernel_5533/766213145.py:105: DtypeWarning: Columns (3,39,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, header=None, names=COL_NAMES)


  Loaded: 700,001 rows
  ✅ Done — 700,001 rows written

Processing file 3/4...
  Loaded: 700,001 rows
  ✅ Done — 700,001 rows written

Processing file 4/4...
  Loaded: 440,044 rows
  ✅ Done — 440,044 rows written

✅ Processing complete!
   CSV  → /content/output/unsw_nb15_processed.csv
   JSON → /content/output/unsw_nb15_processed.json

Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>